# K562 H3K27ac Enhancer Activity Regression with HyenaDNA
## Predicting Enhancer Strength with a State-Space Model

**Companion to the NT regression notebook -- same task, different architecture.**

---

### Connection to Previous Notebooks

| Notebook | Task | Model | Architecture |
|----------|------|-------|-------------|
| **Regression** (NT) | How active is this enhancer? | Nucleotide Transformer 500M | Transformer (encoder) |
| **Classification** (NT) | Is this an enhancer? | Nucleotide Transformer 500M | Transformer (encoder) |
| **Classification** (HyenaDNA) | Is this an enhancer? | HyenaDNA medium-160k | State-space model |
| **This notebook** | **How active is this enhancer?** | **HyenaDNA medium-160k** | **State-space model** |

### Why HyenaDNA for regression?

In the classification notebook, HyenaDNA showed competitive performance for
binary enhancer detection. Now we ask the harder question:

> *Can HyenaDNA predict **how active** an enhancer is -- not just whether it is one?*

Regression is more demanding than classification because the model must capture
**quantitative** differences in enhancer activity, not just a binary boundary.
This tests whether HyenaDNA's character-level representations encode enough
information about regulatory grammar to predict continuous signal intensity.

### What stays the same?
- Raw data (ENCODE K562 H3K27ac ChIP-seq)
- Chromosome-aware splits (train chr1-16, val chr17-18, test chr19-22+X)
- CNN baseline
- Staged fine-tuning strategy
- Log1p signal normalization

### Lecture Alignment

| Lecture Step | Section | Notes |
|:---:|---|---|
| **0** | Biological question | Regression: predict continuous H3K27ac signal |
| **1** | Model selection | HyenaDNA medium (14.2M params, SSM) |
| **2** | Fine-tuning strategy | Same staged approach |
| **3** | Data curation | Peaks only (no negatives needed for regression) |
| **4-5** | Tokenization | Character-level (1 nt = 1 token) |
| **6** | Experimental setup | MSE loss, Pearson/Spearman/RMSE |
| **7** | Fine-tuning | Frozen -> partial unfreeze |
| **8** | Evaluation | Scatter plots, per-chromosome, saliency |

## HyenaDNA Recap

> See the HyenaDNA classification notebook for the full architecture discussion.

### Key properties relevant to regression

| Feature | HyenaDNA | Nucleotide Transformer |
|---------|:---:|:---:|
| **Architecture** | State-space model (Hyena operator) | Transformer (attention) |
| **Parameters** | 14.2M | 500M |
| **Tokenization** | Character-level (1 nt = 1 token) | Learned (~170 tokens/kb) |
| **Pooling** | Mean pooling (no CLS token) | CLS token |
| **Forward pass** | `input_ids` only (no attention_mask) | `input_ids` + `attention_mask` |
| **Output** | Tuple (use `out[0]`) | Named object (use `out.last_hidden_state`) |

### Why might regression be different from classification?

- **Classification** only needs to find features that separate enhancers from
  non-enhancers -- a relatively coarse distinction
- **Regression** must capture the **quantitative grammar** of enhancer strength --
  subtle sequence features that modulate activity levels
- The model needs to learn that certain motif combinations, spacings, and contexts
  produce stronger vs weaker enhancer activity

In [ ]:
# === INSTALLATION & DATA DOWNLOAD ===# Same data as previous notebooks#!pip install -q torch transformers pyBigWig pandas numpy scikit-learn tqdm matplotlib seaborn captum pyfaidx einopsimport osif not os.path.exists('K562_H3K27ac.bigWig'):    !wget -qO K562_H3K27ac.bigWig "https://www.encodeproject.org/files/ENCFF465GBD/@@download/ENCFF465GBD.bigWig"if not os.path.exists('K562_peaks.bed'):    !wget -qO K562_peaks.bed.gz "https://www.encodeproject.org/files/ENCFF038DDS/@@download/ENCFF038DDS.bed.gz" && gunzip -f K562_peaks.bed.gzif not os.path.exists('hg38.fa'):    !wget -qO hg38.fa.gz "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz" && gunzip -f hg38.fa.gz && samtools faidx hg38.faprint('Data files ready.')

In [ ]:
import torchimport torch.nn as nnimport numpy as npimport pandas as pdimport pyBigWigfrom torch.utils.data import Dataset, DataLoaderfrom transformers import AutoTokenizer, AutoModelfrom scipy.stats import pearsonr, spearmanrfrom sklearn.metrics import mean_squared_error, r2_scorefrom tqdm import tqdmimport matplotlib.pyplot as pltimport seaborn as snsfrom captum.attr import IntegratedGradientsfrom pyfaidx import Fastadevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Device: {device}')

In [ ]:
# ============================================================# CONFIGURATION# ============================================================QUICK_MODE = FalseHYENA_MODEL = "LongSafari/hyenadna-medium-160k-seqlen-hf"HYENA_MAX_LENGTH = 1026  # 1000bp + special tokensif QUICK_MODE:    MAX_TRAIN, MAX_VAL, MAX_TEST = 5000, 1000, 2000    NUM_EPOCHS = 5    BATCH_SIZE = 32    print("QUICK_MODE: using data subset")else:    MAX_TRAIN, MAX_VAL, MAX_TEST = None, None, None    NUM_EPOCHS = 10    BATCH_SIZE = 32    print("FULL MODE: using complete dataset")NUM_UNFREEZE_LAYERS = 2PATIENCE = 3

## Step 0 -- Define the Biological Question

Same as the NT regression notebook:

> Given a 1 kb DNA sequence centered on a candidate enhancer summit in K562 cells,
> predict the mean H3K27ac signal intensity over that window.

| Aspect | Definition |
|--------|-----------|
| **Input** | 1 kb DNA sequence |
| **Output** | Continuous H3K27ac signal (log1p transformed) |
| **Resolution** | Locus-level, K562 |

### What we're testing

> *Can a 14M-parameter state-space model predict quantitative enhancer activity
> as well as a 500M-parameter transformer?*

In [ ]:
# ============================================================# Steps 0+3: Load data, extract signals and sequences# (Same as NT regression notebook -- peaks only, no negatives)# ============================================================bw = pyBigWig.open('K562_H3K27ac.bigWig')peaks = pd.read_csv('K562_peaks.bed', sep='\t', header=None, comment='#')peaks.columns = ['chrom','start','end','name','score','strand','signal','pval','qval','peak']peaks['summit'] = peaks['start'] + peaks['peak']peaks['win_start'] = peaks['summit'] - 500peaks['win_end'] = peaks['summit'] + 500valid_chroms = ['chr' + str(i) for i in range(1, 23)] + ['chrX']peaks = peaks[peaks.chrom.isin(valid_chroms)]peaks = peaks[peaks.win_start >= 0]# Extract mean H3K27ac signal (regression target)peaks['mean_signal'] = peaks.apply(    lambda r: np.nanmean(bw.values(r.chrom, r.win_start, r.win_end)), axis=1)peaks = peaks.dropna(subset=['mean_signal']).query('mean_signal > 0')genome = Fasta('hg38.fa')def get_seq(chrom, start, end):    try: return str(genome[chrom][start:end]).upper()    except: return Nonepeaks['sequence'] = peaks.apply(    lambda r: get_seq(r.chrom, r.win_start, r.win_end), axis=1)peaks = peaks.dropna(subset=['sequence'])peaks = peaks[peaks.sequence.apply(lambda s: s.count('N')/len(s) < 0.1)]peaks = peaks.reset_index(drop=True)print(f'Total enhancer windows: {len(peaks):,}')print(f'Signal range: {peaks.mean_signal.min():.2f} -- {peaks.mean_signal.max():.2f}')

In [ ]:
# ============================================================# Step 3: Chromosome-aware splits + log1p transform# ============================================================train_chroms = ['chr' + str(i) for i in range(1, 17)]val_chroms   = ['chr17', 'chr18']test_chroms  = ['chr' + str(i) for i in range(19, 23)] + ['chrX']train_df = peaks[peaks.chrom.isin(train_chroms)].copy()val_df   = peaks[peaks.chrom.isin(val_chroms)].copy()test_df  = peaks[peaks.chrom.isin(test_chroms)].copy()# Log1p transform (same as NT regression notebook)for df in [train_df, val_df, test_df]:    df['log_signal'] = np.log1p(df['mean_signal'].values)print(f'Splits -- Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')print(f'Log-signal stats:')for nm, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:    print(f'  {nm}: mean={df.log_signal.mean():.2f}, std={df.log_signal.std():.2f}')if QUICK_MODE:    if MAX_TRAIN and len(train_df) > MAX_TRAIN:        train_df = train_df.sample(n=MAX_TRAIN, random_state=42)    if MAX_VAL and len(val_df) > MAX_VAL:        val_df = val_df.sample(n=MAX_VAL, random_state=42)    if MAX_TEST and len(test_df) > MAX_TEST:        test_df = test_df.sample(n=MAX_TEST, random_state=42)    print(f'After QUICK_MODE -- Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

## Step 1 -- Model Selection

> *Lecture: "Always compare to a reasonable baseline, not just other FMs."*

### HyenaDNA for regression

The same HyenaDNA medium model as the classification notebook, but now with a
**regression head** (outputting a continuous value instead of a logit).

| Component | Classification Notebook | This Notebook (Regression) |
|-----------|:---:|:---:|
| **Backbone** | HyenaDNA medium | HyenaDNA medium |
| **Head output** | 1 logit (-> sigmoid) | 1 continuous value |
| **Loss** | BCEWithLogitsLoss | **MSE** |
| **Metrics** | AUROC, AUPRC | **Pearson, Spearman, RMSE** |
| **Data** | Peaks + negatives | **Peaks only** |

In [ ]:
# ============================================================# Step 1: HyenaDNA Regressor# ============================================================# Same backbone as classification, different head purpose.# Key HyenaDNA properties:#   - trust_remote_code=True required#   - No attention_mask (SSM processes sequentially)#   - Mean pooling (no CLS token)#   - Output is tuple: out[0] = hidden statesclass HyenaDNARegressor(nn.Module):    """HyenaDNA backbone + regression head.    Uses mean pooling over sequence positions.    Output: continuous predicted signal value.    """    def __init__(self, model_name, freeze_backbone=False):        super().__init__()        self.backbone = AutoModel.from_pretrained(            model_name, trust_remote_code=True        )        if freeze_backbone:            for p in self.backbone.parameters():                p.requires_grad = False        d_model = self.backbone.config.d_model  # 256 for medium        self.head = nn.Sequential(            nn.Linear(d_model, 128),            nn.ReLU(),            nn.Dropout(0.1),            nn.Linear(128, 1),        )    def forward(self, input_ids):        hidden = self.backbone(input_ids)[0]  # (batch, seq_len, d_model)        pooled = hidden.mean(dim=1)            # mean pooling        return self.head(pooled).squeeze(-1)# Inspect architecture_temp = AutoModel.from_pretrained(HYENA_MODEL, trust_remote_code=True)print(f'HyenaDNA-medium: d_model={_temp.config.d_model}, n_layer={_temp.config.n_layer}')print(f'Parameters: {sum(p.numel() for p in _temp.parameters()):,}')del _temp

In [ ]:
# ============================================================# Step 1: CNN Baseline Regressor (same as NT regression notebook)# ============================================================def one_hot_encode(seq, length=1000):    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}    encoded = np.zeros((4, length), dtype=np.float32)    for i, base in enumerate(seq[:length]):        if base in mapping:            encoded[mapping[base], i] = 1.0    return encodedclass CNNBaseline(nn.Module):    def __init__(self):        super().__init__()        self.conv = nn.Sequential(            nn.Conv1d(4, 64, kernel_size=15, padding=7),            nn.ReLU(), nn.BatchNorm1d(64), nn.MaxPool1d(4),            nn.Conv1d(64, 128, kernel_size=7, padding=3),            nn.ReLU(), nn.BatchNorm1d(128), nn.MaxPool1d(4),            nn.Conv1d(128, 64, kernel_size=5, padding=2),            nn.ReLU(), nn.AdaptiveAvgPool1d(1),        )        self.head = nn.Sequential(            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1), nn.Linear(32, 1),        )    def forward(self, x):        return self.head(self.conv(x).squeeze(-1)).squeeze(-1)

## Step 2 -- Fine-Tuning Strategy

Same staged approach:

1. **Stage 1 -- Frozen backbone:** train regression head only
2. **Stage 2 -- Partial unfreeze:** top 2 Hyena layers + differential LR

> *With only 14.2M parameters (vs NT's 500M), HyenaDNA is at lower risk of
> overfitting when partially unfrozen. Expect more benefit from Stage 2.*

## Steps 4-5 -- Tokenization & Preprocessing

### Character-level tokenization (same as HyenaDNA classification)

Each nucleotide becomes one token. A 1 kb sequence = ~1000 tokens.

### Regression-specific preprocessing

| Aspect | Classification | Regression |
|--------|:---:|:---:|
| **Labels** | Binary (0/1) | log1p(mean_signal) |
| **Normalization** | Not needed | **log1p transform** (reduces skew) |
| **Loss** | BCE | **MSE** |

> *Lecture: "Log-transform or z-score normalize if needed"*

In [ ]:
# ============================================================# Steps 4-5: HyenaDNA tokenizer and datasets# ============================================================hyena_tokenizer = AutoTokenizer.from_pretrained(    HYENA_MODEL, trust_remote_code=True)print(f'HyenaDNA vocabulary size: {hyena_tokenizer.vocab_size}')# Verify token counttest_enc = hyena_tokenizer("A" * 1000, truncation=True, max_length=HYENA_MAX_LENGTH)print(f'1000bp sequence -> {len(test_enc["input_ids"])} tokens (character-level)')class HyenaRegressionDataset(Dataset):    """Dataset for HyenaDNA: character-level tokens + continuous label."""    def __init__(self, df):        self.seqs = df.sequence.values        self.labels = df.log_signal.values.astype(np.float32)    def __len__(self):        return len(self.seqs)    def __getitem__(self, idx):        enc = hyena_tokenizer(            self.seqs[idx], return_tensors='pt',            truncation=True, padding='max_length',            max_length=HYENA_MAX_LENGTH,        )        return {            'input_ids': enc.input_ids.squeeze(0),            'labels': torch.tensor(self.labels[idx], dtype=torch.float32),        }class CNNRegressionDataset(Dataset):    def __init__(self, df, seq_len=1000):        self.seqs = df.sequence.values        self.labels = df.log_signal.values.astype(np.float32)        self.seq_len = seq_len    def __len__(self):        return len(self.seqs)    def __getitem__(self, idx):        return {            'sequence': torch.tensor(one_hot_encode(self.seqs[idx], self.seq_len)),            'labels': torch.tensor(self.labels[idx], dtype=torch.float32),        }# DataLoaderstrain_loader_hyena = DataLoader(HyenaRegressionDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)val_loader_hyena   = DataLoader(HyenaRegressionDataset(val_df),   batch_size=BATCH_SIZE)test_loader_hyena  = DataLoader(HyenaRegressionDataset(test_df),  batch_size=BATCH_SIZE)train_loader_cnn = DataLoader(CNNRegressionDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)val_loader_cnn   = DataLoader(CNNRegressionDataset(val_df),   batch_size=BATCH_SIZE)test_loader_cnn  = DataLoader(CNNRegressionDataset(test_df),  batch_size=BATCH_SIZE)print(f'Hyena -- Train: {len(train_loader_hyena)} batches | Val: {len(val_loader_hyena)} | Test: {len(test_loader_hyena)}')print(f'CNN   -- Train: {len(train_loader_cnn)} batches | Val: {len(val_loader_cnn)} | Test: {len(test_loader_cnn)}')

## Step 6 -- Experimental Setup

### Regression metrics (same as NT regression notebook)

| Metric | What it measures |
|--------|-----------------|
| **Pearson r** | Linear correlation between predicted and actual |
| **Spearman rho** | Rank correlation (robust to outliers) |
| **RMSE** | Error magnitude in log-signal units |
| **R-squared** | Fraction of variance explained |

### Loss and early stopping

- **Loss:** MSE (Mean Squared Error)
- **Early stopping:** on validation Pearson r

In [ ]:
# ============================================================# Step 6: Training and evaluation utilities# ============================================================def train_one_epoch(model, loader, optimizer, loss_fn, model_type='hyena'):    model.train()    total_loss = 0    for batch in tqdm(loader, desc='Training', leave=False):        optimizer.zero_grad()        labels = batch['labels'].to(device)        if model_type == 'hyena':            preds = model(batch['input_ids'].to(device))        else:            preds = model(batch['sequence'].to(device))        loss = loss_fn(preds, labels)        loss.backward()        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)        optimizer.step()        total_loss += loss.item()    return total_loss / len(loader)def evaluate(model, loader, model_type='hyena'):    model.eval()    all_preds, all_labels = [], []    with torch.no_grad():        for batch in loader:            if model_type == 'hyena':                preds = model(batch['input_ids'].to(device))            else:                preds = model(batch['sequence'].to(device))            all_preds.append(preds.cpu())            all_labels.append(batch['labels'])    preds = torch.cat(all_preds).numpy()    labels = torch.cat(all_labels).numpy()    return {        'pearson':  float(pearsonr(labels, preds)[0]),        'spearman': float(spearmanr(labels, preds).correlation),        'rmse':     float(np.sqrt(mean_squared_error(labels, preds))),        'r2':       float(r2_score(labels, preds)),    }def get_predictions(model, loader, model_type='hyena'):    model.eval()    all_preds, all_labels = [], []    with torch.no_grad():        for batch in loader:            if model_type == 'hyena':                preds = model(batch['input_ids'].to(device))            else:                preds = model(batch['sequence'].to(device))            all_preds.append(preds.cpu())            all_labels.append(batch['labels'])    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()def train_model(model, train_ldr, val_ldr, optimizer, loss_fn,                model_type='hyena', num_epochs=NUM_EPOCHS, patience=PATIENCE,                save_path='best.pt'):    best_pearson = -float('inf')    patience_ctr = 0    history = {'train_loss': [], 'val_pearson': [], 'val_spearman': []}    for epoch in range(num_epochs):        loss = train_one_epoch(model, train_ldr, optimizer, loss_fn, model_type)        val = evaluate(model, val_ldr, model_type)        history['train_loss'].append(loss)        history['val_pearson'].append(val['pearson'])        history['val_spearman'].append(val['spearman'])        print(f'  Epoch {epoch+1}/{num_epochs} -- Loss: {loss:.4f} | '              f'Val Pearson: {val["pearson"]:.4f} | Val Spearman: {val["spearman"]:.4f}')        if val['pearson'] > best_pearson:            best_pearson = val['pearson']            torch.save(model.state_dict(), save_path)            patience_ctr = 0        else:            patience_ctr += 1            if patience_ctr >= patience:                print(f'  Early stopping at epoch {epoch+1}')                break    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))    return model, history

## Step 7 -- Fine-Tuning

### Stage 1: Frozen Backbone

> *Lecture: "Freeze backbone, train prediction head -> diagnostic baseline"*

In [ ]:
# ============================================================# Step 7, Stage 1: Frozen backbone# ============================================================print('=' * 60)print('STAGE 1: Frozen HyenaDNA Backbone')print('=' * 60)hyena_model = HyenaDNARegressor(HYENA_MODEL, freeze_backbone=True).to(device)trainable = sum(p.numel() for p in hyena_model.parameters() if p.requires_grad)total_params = sum(p.numel() for p in hyena_model.parameters())print(f'Trainable: {trainable:,} / {total_params:,} ({100*trainable/total_params:.2f}%)')optimizer_s1 = torch.optim.AdamW(    filter(lambda p: p.requires_grad, hyena_model.parameters()),    lr=1e-3, weight_decay=0.01)loss_fn = nn.MSELoss()hyena_model, history_s1 = train_model(    hyena_model, train_loader_hyena, val_loader_hyena, optimizer_s1, loss_fn,    model_type='hyena', save_path='best_hyena_reg_frozen.pt')frozen_test = evaluate(hyena_model, test_loader_hyena, model_type='hyena')print(f'\nStage 1 (Frozen) Test Results:')for k, v in frozen_test.items(): print(f'  {k}: {v:.4f}')

### Stage 2: Partial Unfreeze

> *Lecture: "Unfreeze top N layers, train with low LR on backbone"*

HyenaDNA-medium has 8 Hyena layers. We unfreeze the top 2 with differential LR.

In [ ]:
# ============================================================# Step 7, Stage 2: Unfreeze top layers# ============================================================print('=' * 60)print(f'STAGE 2: Unfreezing Top {NUM_UNFREEZE_LAYERS} Layers')print('=' * 60)hyena_model.load_state_dict(    torch.load('best_hyena_reg_frozen.pt', map_location=device, weights_only=True))# Discover layer structurehyena_layers = Nonefor name, module in hyena_model.backbone.named_modules():    if isinstance(module, nn.ModuleList) and len(list(module)) > 2:        hyena_layers = list(module)        print(f'Found {len(hyena_layers)} layers at: backbone.{name}')        breakif hyena_layers is None:    print('Could not auto-detect layers -- unfreezing all backbone parameters')    for p in hyena_model.backbone.parameters():        p.requires_grad = True    unfrozen_params = list(hyena_model.backbone.parameters())else:    print(f'Unfreezing top {NUM_UNFREEZE_LAYERS} of {len(hyena_layers)} layers')    unfrozen_params = []    for layer in hyena_layers[-NUM_UNFREEZE_LAYERS:]:        for p in layer.parameters():            p.requires_grad = True        unfrozen_params.extend(layer.parameters())trainable = sum(p.numel() for p in hyena_model.parameters() if p.requires_grad)print(f'Trainable: {trainable:,} / {total_params:,} ({100*trainable/total_params:.2f}%)')optimizer_s2 = torch.optim.AdamW([    {'params': hyena_model.head.parameters(), 'lr': 1e-3},    {'params': unfrozen_params, 'lr': 1e-5},], weight_decay=0.01)hyena_model, history_s2 = train_model(    hyena_model, train_loader_hyena, val_loader_hyena, optimizer_s2, loss_fn,    model_type='hyena', save_path='best_hyena_reg_partial.pt')partial_test = evaluate(hyena_model, test_loader_hyena, model_type='hyena')print(f'\nStage 2 (Partial FT) Test Results:')for k, v in partial_test.items(): print(f'  {k}: {v:.4f}')

### CNN Baseline

> *Lecture: "Always compare to a reasonable baseline, not just other FMs."*

In [ ]:
# ============================================================# Step 7: CNN Baseline# ============================================================print('=' * 60)print('CNN BASELINE')print('=' * 60)cnn_model = CNNBaseline().to(device)print(f'CNN parameters: {sum(p.numel() for p in cnn_model.parameters()):,}')optimizer_cnn = torch.optim.AdamW(cnn_model.parameters(), lr=1e-3, weight_decay=0.01)cnn_model, history_cnn = train_model(    cnn_model, train_loader_cnn, val_loader_cnn, optimizer_cnn, loss_fn,    model_type='cnn', save_path='best_hyena_reg_cnn.pt')cnn_test = evaluate(cnn_model, test_loader_cnn, model_type='cnn')print(f'\nCNN Baseline Test Results:')for k, v in cnn_test.items(): print(f'  {k}: {v:.4f}')

## Step 8 -- Evaluation & Interpretation

### Evaluation plan

1. **Model comparison table** -- all regression metrics
2. **Predicted vs actual scatter plots** -- visual assessment
3. **Per-chromosome metrics** -- generalization check
4. **Saliency analysis** -- what drives high predictions?
5. **Training curves** -- convergence behavior

In [ ]:
# ============================================================# Step 8: Model comparison# ============================================================results = pd.DataFrame({    'HyenaDNA Frozen': frozen_test,    'HyenaDNA Partial FT': partial_test,    'CNN Baseline': cnn_test,}).Tprint('=' * 60)print('MODEL COMPARISON (Test Set)')print('=' * 60)print(results.to_string(float_format='{:.4f}'.format))fig, axes = plt.subplots(1, 2, figsize=(12, 5))results[['pearson', 'spearman']].plot.barh(ax=axes[0], color=['mediumpurple', 'coral'])axes[0].set_xlabel('Correlation')axes[0].set_title('Correlation Metrics (higher is better)')results[['rmse']].plot.barh(ax=axes[1], color='goldenrod', legend=False)axes[1].set_xlabel('RMSE (log-signal)')axes[1].set_title('RMSE (lower is better)')plt.tight_layout()plt.show()

In [ ]:
# ============================================================# Step 8: Predicted vs Actual scatter plots# ============================================================hyena_frozen = HyenaDNARegressor(HYENA_MODEL, freeze_backbone=True).to(device)hyena_frozen.load_state_dict(    torch.load('best_hyena_reg_frozen.pt', map_location=device, weights_only=True))fig, axes = plt.subplots(1, 3, figsize=(16, 5))for ax, (title, mdl, loader, mtype) in zip(axes, [    ('HyenaDNA Frozen', hyena_frozen, test_loader_hyena, 'hyena'),    ('HyenaDNA Partial FT', hyena_model, test_loader_hyena, 'hyena'),    ('CNN Baseline', cnn_model, test_loader_cnn, 'cnn'),]):    preds, labels = get_predictions(mdl, loader, mtype)    ax.scatter(labels, preds, alpha=0.3, s=10, color='mediumpurple')    lims = [min(labels.min(), preds.min()), max(labels.max(), preds.max())]    ax.plot(lims, lims, 'r--', alpha=0.5)    ax.set_xlabel('Actual log(signal)')    ax.set_ylabel('Predicted log(signal)')    r = pearsonr(labels, preds)[0]    ax.set_title(f'{title}\nPearson r = {r:.3f}')    ax.set_aspect('equal', adjustable='box')plt.suptitle('Predicted vs Actual Enhancer Activity', fontsize=14)plt.tight_layout()plt.show()

In [ ]:
# ============================================================# Step 8: Per-chromosome evaluation (using TRAINED model)# ============================================================print('Per-Chromosome Test Metrics (HyenaDNA Partial FT)')print('-' * 50)chrom_results = {}for chrom in test_chroms:    subset = test_df[test_df.chrom == chrom]    if len(subset) < 10: continue    ldr = DataLoader(HyenaRegressionDataset(subset), batch_size=BATCH_SIZE)    metrics = evaluate(hyena_model, ldr, model_type='hyena')    chrom_results[chrom] = metrics    print(f'  {chrom} (n={len(subset):,}): Pearson={metrics["pearson"]:.3f}, '          f'Spearman={metrics["spearman"]:.3f}, RMSE={metrics["rmse"]:.3f}')if chrom_results:    chrom_df = pd.DataFrame(chrom_results).T    fig, ax = plt.subplots(figsize=(10, 5))    chrom_df[['pearson', 'spearman']].plot.bar(ax=ax, color=['mediumpurple', 'coral'])    ax.axhline(y=partial_test['pearson'], color='mediumpurple', linestyle='--', alpha=0.5)    ax.set_ylabel('Correlation')    ax.set_title('Per-Chromosome Performance (HyenaDNA Partial FT)')    plt.xticks(rotation=45)    plt.legend()    plt.tight_layout()    plt.show()

### Saliency / Attribution Analysis

> *Lecture: "Do high-saliency regions overlap known enhancer motifs?"*

Integrated Gradients on the CNN baseline -- same approach as all other notebooks.
For regression, we examine what drives the model to predict **high** activity.

In [ ]:
# ============================================================# Step 8: Saliency (Integrated Gradients on CNN)# ============================================================cnn_model.eval()# Select a high-activity test examplehigh_idx = test_df.log_signal.values.argmax()sample_row = test_df.iloc[high_idx]sample_encoded = torch.tensor(    one_hot_encode(sample_row.sequence), dtype=torch.float32).unsqueeze(0).to(device)sample_encoded.requires_grad_(True)ig = IntegratedGradients(cnn_model)attributions = ig.attribute(sample_encoded, n_steps=50)importance = np.abs(attributions.squeeze().cpu().detach().numpy()).sum(axis=0)smoothed = np.convolve(importance, np.ones(20)/20, mode='same')fig, axes = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={'height_ratios': [2, 1]})pred_val = cnn_model(sample_encoded).item()axes[0].fill_between(range(len(smoothed)), smoothed, alpha=0.7, color='mediumpurple')axes[0].set_ylabel('Attribution (smoothed)')axes[0].set_title(    f'Integrated Gradients (CNN Baseline)\n'    f'{sample_row.chrom}:{sample_row.win_start}-{sample_row.win_end} | '    f'Predicted: {pred_val:.2f} | Actual: {sample_row.log_signal:.2f}')axes[0].set_xlim(0, len(smoothed))gc = np.array([1 if b in 'GC' else 0 for b in sample_row.sequence[:1000]])gc_smooth = np.convolve(gc, np.ones(20)/20, mode='same')axes[1].fill_between(range(len(gc_smooth)), gc_smooth, alpha=0.5, color='coral')axes[1].set_ylabel('GC Content')axes[1].set_xlabel('Position (bp)')axes[1].set_xlim(0, len(gc_smooth))axes[1].set_ylim(0, 1)plt.tight_layout()plt.show()

In [ ]:
# ============================================================# Training curves# ============================================================fig, axes = plt.subplots(1, 3, figsize=(16, 4))for ax, (title, hist) in zip(axes, [    ('HyenaDNA Stage 1 (Frozen)', history_s1),    ('HyenaDNA Stage 2 (Partial FT)', history_s2),    ('CNN Baseline', history_cnn),]):    epochs = range(1, len(hist['train_loss']) + 1)    ax2 = ax.twinx()    ax.plot(epochs, hist['train_loss'], 'b-o', markersize=4, label='Train Loss')    ax2.plot(epochs, hist['val_pearson'], 'r-s', markersize=4, label='Val Pearson')    ax2.plot(epochs, hist['val_spearman'], 'g-^', markersize=4, label='Val Spearman')    ax.set_xlabel('Epoch')    ax.set_ylabel('MSE Loss', color='b')    ax2.set_ylabel('Correlation', color='r')    ax.set_title(title)    l1, lb1 = ax.get_legend_handles_labels()    l2, lb2 = ax2.get_legend_handles_labels()    ax.legend(l1+l2, lb1+lb2, loc='center right', fontsize=8)plt.tight_layout()plt.show()

## Cross-Model Comparison: Regression

After running both regression notebooks, compare results:

### Results template

| Model | Params | Tokenization | Test Pearson | Test Spearman | Test RMSE |
|-------|-------:|-------------|:---:|:---:|:---:|
| CNN Baseline | 105K | One-hot | ___ | ___ | ___ |
| **NT-500M** (frozen) | 500M | 6-mer | ___ | ___ | ___ |
| **NT-500M** (partial FT) | 500M | 6-mer | ___ | ___ | ___ |
| **HyenaDNA-medium** (frozen) | 14.2M | Character | ___ | ___ | ___ |
| **HyenaDNA-medium** (partial FT) | 14.2M | Character | ___ | ___ | ___ |

### Discussion questions

1. **Regression vs classification:** Is the performance gap between HyenaDNA and
   NT larger or smaller for regression than for classification? What does this
   tell us about the models' representations?

2. **Does model size matter more for regression?** Regression requires finer-grained
   predictions. Does the 500M-parameter NT have a larger advantage over 14M-parameter
   HyenaDNA for regression than for classification?

3. **Character-level vs learned tokenization:** Does preserving single-nucleotide
   resolution (HyenaDNA) help with quantitative predictions, or does the compressed
   representation from learned tokens (NT) work as well?

4. **Staged fine-tuning:** Does partial unfreezing help HyenaDNA more for regression
   than for classification? The regression task may require more adaptation of the
   pre-trained representations.

## Summary

### What's Different from the NT Regression Notebook

| Aspect | NT Regression | This Notebook |
|--------|:---:|:---:|
| **Model** | NT-500M (transformer) | HyenaDNA medium (SSM) |
| **Parameters** | 500M | 14.2M |
| **Tokenization** | Learned 6-mer | Character-level |
| **Pooling** | CLS token | Mean pooling |
| **Forward pass** | input_ids + attention_mask | input_ids only |
| **Everything else** | Same | Same |

### Key Takeaways

1. **The regression workflow is model-agnostic** -- swapping HyenaDNA for NT
   required changing only the model class and tokenizer, not the training loop,
   metrics, or evaluation code.

2. **Regression is harder than classification** -- both models may show lower
   Pearson correlation on regression than AUROC on classification, because
   predicting continuous values is inherently more challenging.

3. **Model size vs task complexity** -- for the relatively short (1 kb) sequences
   in this task, the smaller HyenaDNA may be surprisingly competitive, suggesting
   that model size matters less than pre-training data and architecture fit.

### The Complete Notebook Series

| # | Notebook | Task | Model |
|---|---------|------|-------|
| 1 | Regression | How active? | NT-500M |
| 2 | Classification | Enhancer or not? | NT-500M |
| 3 | Classification | Enhancer or not? | HyenaDNA |
| 4 | Classification | Enhancer or not? | DNABERT-2 |
| **5** | **Regression** | **How active?** | **HyenaDNA** |